In [1]:
"""
04_feature_engineering.py
--------------------------
Builds the final modeling table: one row per customer, with
RFM features + demographics (from the calibration window) and
the target variable (total spend in the holdout window).

Reads:
    data/processed/eligible_customers.csv
    data/processed/calibration_transactions.csv
    data/processed/holdout_transactions.csv

Writes:
    data/processed/customer_features.csv
"""

import pandas as pd

# ----------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------
customers = pd.read_csv(
    "data/processed/eligible_customers.csv",
    parse_dates=["acquisition_date", "calibration_end_date", "holdout_end_date"],
)
calibration_tx = pd.read_csv("data/processed/calibration_transactions.csv", parse_dates=["transaction_date"])
holdout_tx = pd.read_csv("data/processed/holdout_transactions.csv", parse_dates=["transaction_date"])

print(f"Eligible customers: {len(customers)}")

# ----------------------------------------------------------------
# 2. RFM FEATURES - calculated ONLY from calibration-window data
# ----------------------------------------------------------------
rfm = calibration_tx.groupby("customer_id").agg(
    frequency=("amount", "count"),
    monetary_total=("amount", "sum"),
    monetary_avg=("amount", "mean"),
    last_purchase_date=("transaction_date", "max"),
).reset_index()

# Merge to get each customer's calibration_end_date for the recency calc
rfm = rfm.merge(customers[["customer_id", "calibration_end_date"]], on="customer_id")
rfm["recency_days"] = (rfm["calibration_end_date"] - rfm["last_purchase_date"]).dt.days
rfm = rfm.drop(columns=["last_purchase_date", "calibration_end_date"])

# ----------------------------------------------------------------
# 3. TARGET VARIABLE - total spend in the holdout window
#    (customers with NO holdout purchases get a target of 0 -
#    this is a real, important case, not missing data)
# ----------------------------------------------------------------
target = holdout_tx.groupby("customer_id")["amount"].sum().reset_index()
target.columns = ["customer_id", "target_holdout_spend"]

# ----------------------------------------------------------------
# 4. COMBINE INTO ONE MODELING TABLE
# ----------------------------------------------------------------
features = customers[["customer_id", "acquisition_channel", "age_band", "region"]].merge(
    rfm, on="customer_id", how="left"
)
features = features.merge(target, on="customer_id", how="left")

# Customers with zero calibration purchases shouldn't exist (everyone
# has at least their acquisition-day purchase) - but target NaN means
# zero holdout purchases, which is real and should be filled as 0.
features["target_holdout_spend"] = features["target_holdout_spend"].fillna(0)

print(f"\nFinal feature table shape: {features.shape}")
print(f"Columns: {list(features.columns)}\n")

# ----------------------------------------------------------------
# 5. SANITY CHECKS
# ----------------------------------------------------------------
print("Feature summary statistics:")
print(features[["frequency", "monetary_total", "monetary_avg", "recency_days", "target_holdout_spend"]].describe().round(2))

print("\nQuick correlation check - does calibration behavior predict the target?")
correlations = features[["frequency", "monetary_total", "monetary_avg", "recency_days", "target_holdout_spend"]].corr()["target_holdout_spend"]
print(correlations.round(3))

print("\nAverage target (future spend) by acquisition channel - should still show Referral leading:")
print(features.groupby("acquisition_channel")["target_holdout_spend"].mean().sort_values(ascending=False).round(2))

# ----------------------------------------------------------------
# 6. SAVE OUTPUT
# ----------------------------------------------------------------
features.to_csv("data/processed/customer_features.csv", index=False)
print("\nSaved: data/processed/customer_features.csv")

Eligible customers: 4000

Final feature table shape: (4000, 9)
Columns: ['customer_id', 'acquisition_channel', 'age_band', 'region', 'frequency', 'monetary_total', 'monetary_avg', 'recency_days', 'target_holdout_spend']

Feature summary statistics:
       frequency  monetary_total  monetary_avg  recency_days  \
count    4000.00         4000.00       4000.00       4000.00   
mean        2.77         3462.22       1196.05         28.90   
std         1.10         2171.57        302.77         23.95   
min         1.00          442.48        333.70          0.00   
25%         2.00         2140.22        984.01         10.00   
50%         3.00         2978.61       1152.00         23.00   
75%         3.00         3932.20       1350.55         41.00   
max         8.00        15732.56       2397.26         90.00   

       target_holdout_spend  
count               4000.00  
mean                4339.18  
std                 4022.62  
min                    0.00  
25%                 1588